# 🌍 TÜBİTAK 2209-A: Yapay Zeka Tabanlı Hava Kalitesi Modeli
## Veri Keşfi, Açıklanabilirlik ve Makine Öğrenimi Raporu (EDA)

Bu notebook, **Kandilli Referans İstasyonu, İstanbul Global Verisi ve Seoul Transfer Öğrenimi Seti'nin** analizini yapıp füzyon (fusion) sonrasındaki başarıyı görselleştirmektedir. Proje raporunda kullanılmak üzere özel olarak tasarlanmıştır.

---

### Hedefler:
1. Birleştirilmiş 160.107 satırlık verinin istatistiksel dağılımı.
2. PM10 ve PM2.5 partiküllerinin kimyasal gazlarla korelasyon analizi (Isı Haritası).
3. Random Forest (Rastgele Orman) algoritmasının özellikleri ne kadar önemsediğinin görselleştirilmesi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from collections import Counter

# Görsel ayarları
plt.style.use('dark_background')
sns.set_palette("Blues_d")

df = pd.read_csv('../fused_training_set.csv')
print(f"✅ Fused Dataset Yüklendi! Toplam Satır: {len(df):,}\n")
display(df.head())

### 1. Hedef Değişkenlerin (PM10 & PM2.5) Dağılımı
Aşağıdaki grafik, modelimizin beslendiği eğitim setinde partikül maddelerin nasıl bir dağılıma sahip olduğunu gösteriyor.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['PM10'], bins=100, kde=True, color='cyan', ax=ax[0])
ax[0].set_title('PM10 Yoğunluğu Dağılımı (µg/m³)')
ax[0].set_xlim(0, 200)

sns.histplot(df['PM25'], bins=100, kde=True, color='teal', ax=ax[1])
ax[1].set_title('PM2.5 Yoğunluğu Dağılımı (µg/m³)')
ax[1].set_xlim(0, 150)

plt.tight_layout()
plt.show()

### 2. Değişkenler Arası Korelasyon Analizi (Heatmap)
Hangi sensör gazının veya meteorolojik olayın PM10'u daha fazla etkilediğini anlamak için korelasyon ısı haritası oluşturuyoruz.

In [ ]:
plt.figure(figsize=(10, 8))
features = ['PM10', 'PM25', 'CO', 'O3', 'NO2', 'SO2', 'Humidity', 'Temperature', 'Wind_Speed']
corr_matrix = df[features].corr()

# Heatmap maskesi (üst üçgeni gizlemek için)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="mako", 
            vmin=-0.2, vmax=1.0, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Sensör Ölçümleri Arası Korelasyon Haritası', fontsize=16, pad=20)
plt.show()

### 3. Yapay Zeka Model Performansı ve Özellik Önemi (Feature Importance)
Veri füzyonu sonrası eğittiğimiz **Random Forest** (Rastgele Orman) modellerinin (PM10 ve PM2.5 için) hangi girdilere daha çok değer verdiğini inceliyoruz. Raporunuzda bu grafik modelinizin "Açıklanabilirliğini (Explainable AI)" kanıtlarken jüriden tam not almanızı sağlayacaktır.

In [ ]:
# Eğitilmiş modelin çıktılarını okuyalım
try:
    with open('../model_metrics.json') as f:
        metrics = json.load(f)
except:
    metrics = None
    print("Önce python train_model.py çalıştırılmalı!")

if metrics:
    pm10_imp = metrics['pm10']['feature_importance']
    pm25_imp = metrics['pm25']['feature_importance']
    
    # PM10 Özellik Önemi Çizdir
    features_10 = list(pm10_imp.keys())
    values_10 = list(pm10_imp.values())
    
    plt.figure(figsize=(9, 5))
    sns.barplot(x=values_10, y=features_10, palette="viridis")
    plt.title(f"PM10 Algoritması İçin Özellik Önemi | Model R²: {metrics['pm10']['r2']:.2f}")
    plt.xlabel('Önem Skoru (Importance)')
    plt.ylabel('Sensör Değişkeni')
    plt.show()
    
    # PM2.5 Özellik Önemi Çizdir
    features_25 = list(pm25_imp.keys())
    values_25 = list(pm25_imp.values())
    
    plt.figure(figsize=(9, 5))
    sns.barplot(x=values_25, y=features_25, palette="magma")
    plt.title(f"PM2.5 Algoritması İçin Özellik Önemi | Model R²: {metrics['pm25']['r2']:.2f}")
    plt.xlabel('Önem Skoru (Importance)')
    plt.ylabel('Sensör Değişkeni')
    plt.show()
else:
    print("Metrics bulunamadı.")